# Power Transformer
Box-Cox and Yeo-Johnson transforms on the concrete strength data, checked with skewness, Q-Q plots and the R² score of linear regression.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import make_pipeline

# one look for every chart: Latin Modern font, white background
FONT = dict(family="Latin Modern Roman", size=15)

## Load the data
1030 concrete mixes: 8 ingredient columns (amounts, and `Age` in days) and the target `Strength`.

In [ ]:
df = pd.read_csv("data/concrete_data.csv")
df.head()

In [ ]:
# 1030 rows, 9 columns
df.shape

In [ ]:
# no missing values anywhere
df.isnull().sum()

In [ ]:
# the min row matters: Box-Cox needs every value above 0
df.describe()

In [ ]:
# how many zeros each column has
(df == 0).sum()

## Split first

In [ ]:
X = df.drop(columns=["Strength"])   # 8 inputs
y = df["Strength"]                  # target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## Baseline: linear regression with no transform

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
# R² on the test set
r2_score(y_test, y_pred)

In [ ]:
# 5-fold cross-validation (cv=5 is the default) on the whole data
cross_val_score(LinearRegression(), X, y, scoring="r2", cv=5).mean()

## Is each column normal?
A histogram with a KDE curve, and a Q-Q plot from `scipy.stats.probplot` drawn in Plotly.

In [ ]:
def hist_and_qq(x, title, colour="#4C78A8"):
    """Histogram + KDE on the left, Q-Q plot on the right."""
    x = np.asarray(x, dtype=float)
    fig = make_subplots(1, 2, subplot_titles=[f"{title}: skew {pd.Series(x).skew():.2f}", "Q-Q plot"])
    fig.add_histogram(x=x, histnorm="probability density", nbinsx=30, marker_color=colour, opacity=0.45, row=1, col=1)
    grid = np.linspace(x.min(), x.max(), 300)
    fig.add_scatter(x=grid, y=stats.gaussian_kde(x)(grid), mode="lines", line_color=colour, row=1, col=1)
    # probplot returns (normal quantiles, sorted data) and the best straight line
    (osm, osr), (slope, intercept, _) = stats.probplot(x, dist="norm")
    fig.add_scatter(x=osm, y=osr, mode="markers", marker=dict(color=colour, size=4), row=1, col=2)
    fig.add_scatter(x=osm, y=slope * osm + intercept, mode="lines", line_color="#E45756", row=1, col=2)
    fig.update_layout(template="simple_white", font=FONT, showlegend=False, height=320, width=850)
    fig.show()

for col in X_train.columns:
    hist_and_qq(X_train[col], col)

## Box-Cox
Box-Cox needs values above 0, and three columns contain zeros, so we add a tiny number to every value first.

In [ ]:
pt = PowerTransformer(method="box-cox")
X_train_transformed = pt.fit_transform(X_train + 0.000001)
X_test_transformed = pt.transform(X_test + 0.000001)

# one lambda learned per column
pd.DataFrame({"cols": X_train.columns, "box_cox_lambdas": pt.lambdas_})

In [ ]:
# same linear regression, on the transformed columns
lr = LinearRegression()
lr.fit(X_train_transformed, y_train)
r2_score(y_test, lr.predict(X_test_transformed))

In [ ]:
# cross-validation, the simple way: transform all of X, then cross-validate
X_transformed = PowerTransformer(method="box-cox").fit_transform(X + 0.000001)
cross_val_score(LinearRegression(), X_transformed, y, scoring="r2", cv=5).mean()

In [ ]:
# the safer way: a pipeline refits the transformer inside each fold (no leakage)
pipe = make_pipeline(PowerTransformer(method="box-cox"), LinearRegression())
cross_val_score(pipe, X + 0.000001, y, scoring="r2", cv=5).mean()

In [ ]:
# before and after for every column
X_train_bc = pd.DataFrame(X_train_transformed, columns=X_train.columns)
for col in X_train.columns:
    hist_and_qq(X_train_bc[col], f"{col} after Box-Cox", "#54A24B")

## Yeo-Johnson
The default method. It accepts 0 and negative values, so no tiny number is needed.

In [ ]:
pt1 = PowerTransformer()          # method="yeo-johnson" is the default
X_train_transformed2 = pt1.fit_transform(X_train)
X_test_transformed2 = pt1.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_transformed2, y_train)
print(r2_score(y_test, lr.predict(X_test_transformed2)))

pd.DataFrame({"cols": X_train.columns, "Yeo_Johnson_lambdas": pt1.lambdas_})

In [ ]:
# cross-validation, simple way and pipeline way
X_transformed2 = PowerTransformer().fit_transform(X)
print(cross_val_score(LinearRegression(), X_transformed2, y, scoring="r2", cv=5).mean())
pipe = make_pipeline(PowerTransformer(), LinearRegression())
print(cross_val_score(pipe, X, y, scoring="r2", cv=5).mean())

In [ ]:
X_train_yj = pd.DataFrame(X_train_transformed2, columns=X_train.columns)
for col in X_train.columns:
    hist_and_qq(X_train_yj[col], f"{col} after Yeo-Johnson", "#54A24B")

## Lambdas side by side
Both transformers were fitted on the same training set, so the two columns compare like with like.

In [ ]:
pd.DataFrame({"cols": X_train.columns,
              "box_cox_lambdas": pt.lambdas_,
              "Yeo_Johnson_lambdas": pt1.lambdas_}).round(3)

In [ ]:
# the output is standardised: mean 0, standard deviation 1 (standardize=True is the default)
X_train_yj.describe().loc[["mean", "std"]].round(2)

## Play with lambda
Drag the slider: Box-Cox on `Age` with each λ from −1 to 2, standardised. The red curve is a normal distribution. Watch the skewness reach 0 near the learned λ = 0.067.

In [ ]:
age = X_train["Age"].values.astype(float)
lams = np.round(np.arange(-1, 2.01, 0.1), 2)
edges = np.arange(-6, 6.01, 0.5)
z_grid = np.linspace(-4, 4, 200)

def frame_data(lam):
    t = stats.boxcox(age, lam)
    z = (t - t.mean()) / t.std()
    share = np.histogram(np.clip(z, -5.99, 5.99), edges)[0] / len(z)
    return [go.Bar(x=edges[:-1] + 0.25, y=share, width=0.46, marker_color="#4C78A8"),
            go.Scatter(x=z_grid, y=0.5 * stats.norm.pdf(z_grid), mode="lines", line_color="#E45756")], pd.Series(t).skew()

frames = []
for lam in lams:
    data, skew = frame_data(lam)
    frames.append(go.Frame(data=data, name=str(lam), layout=dict(title=f"λ = {lam}: skewness {skew:.2f}")))
fig = go.Figure(data=frames[20].data, frames=frames, layout=frames[20].layout)
fig.update_layout(template="simple_white", font=FONT, showlegend=False, height=420, width=850,
                  xaxis=dict(range=[-6, 6], title="standardised value"), yaxis=dict(range=[0, 0.5], title="share of rows"),
                  sliders=[dict(active=20, currentvalue=dict(prefix="λ = "),
                                steps=[dict(label=str(l), method="animate",
                                            args=[[str(l)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                                       for l in lams])])
fig.show()

## Scaling does not change linear regression
Standardising the raw columns gives exactly the same R² as no scaling: the model just rescales its coefficients.

In [ ]:
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
r2_score(y_test, pipe.predict(X_test))